In [1]:
# =============================================================================
# MACHINE LEARNING PARA ECONOMISTAS - TRABAJO PRACTICO N°2
# PARTE II: ARBOLES - INCISOS 4 Y 5
# =============================================================================
#
# Este script reproduce la Parte II utilizando la base model-ready del TP1.
# Mantiene exactamente la particion train/test guardada en la columna "conjunto".
#
# Decisiones metodologicas:
#   1. Se usa la misma particion del TP1 (random_state = 444 ya incorporado).
#   2. Se elimina "const" porque CART no utiliza una columna de unos.
#   3. No se estandariza: CART divide por umbrales y no depende de la escala.
#   4. ccp_alpha se selecciona SOLO con train mediante 10-fold CV.
#   5. Test no se utiliza en Parte II: queda reservado para Parte III.
#   6. Se guarda el arbol final para reutilizar exactamente el mismo modelo
#      en la comparacion de Parte III (matriz de confusion, ROC y metricas).
#   7. La comparacion CART-LASSO usa las seis variables que LASSO llevo a cero
#      en el Cuadro 1 de la Parte I del informe del grupo.
#
# =============================================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, GridSearchCV

In [2]:



# =============================================================================
# 0. RUTAS
# =============================================================================

# Misma estructura de rutas que Parte_I.ipynb.
# El notebook puede ejecutarse desde TP_02 o TP_02/Programs.
raiz = Path.cwd()

if raiz.name == "Programs":
    raiz = raiz.parent

carpeta_input = raiz / "Input"
carpeta_output = raiz / "Output"

RUTA_BASE = carpeta_input / "EPH_T126_limpia.csv"

if not RUTA_BASE.is_file():
    raise FileNotFoundError(
        f"No se encontro la base: {RUTA_BASE}. "
        "Ejecuta el notebook desde TP_02 o TP_02/Programs."
    )

# Los resultados de la Parte II se guardan separados de los de la Parte I.
RUTA_SALIDA = carpeta_output / "output_parte_2"
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. RECUPERAR LA MISMA MUESTRA Y PARTICION DEL TP1
# =============================================================================

base = pd.read_csv(RUTA_BASE)

columnas_obligatorias = {"informal", "conjunto"}
faltantes = columnas_obligatorias - set(base.columns)
assert not faltantes, f"Faltan columnas obligatorias: {faltantes}"

train = base.loc[base["conjunto"].eq("train")].copy()
test = base.loc[base["conjunto"].eq("test")].copy()

columnas_excluir = {"informal", "conjunto", "const"}
variables_x = [c for c in base.columns if c not in columnas_excluir]

X_train = train[variables_x].copy()
y_train = train["informal"].astype(int).copy()

# Se recupera test solo para verificar que la particion queda disponible.
# NO se usa para seleccionar ccp_alpha ni para evaluar el arbol en Parte II.
X_test = test[variables_x].copy()
y_test = test["informal"].astype(int).copy()

assert X_train.isna().sum().sum() == 0, "X_train contiene NaN."
assert X_test.isna().sum().sum() == 0, "X_test contiene NaN."

print("=" * 76)
print("BASE Y PARTICION")
print("=" * 76)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Predictores:", X_train.shape[1])
print("Informalidad train:", round(y_train.mean(), 4))
print("Informalidad test :", round(y_test.mean(), 4))


# =============================================================================
# INCISO 4
# ARBOL PODADO Y ELECCION DE ccp_alpha MEDIANTE 10-FOLD CV
# =============================================================================

# 2. Cost-complexity pruning path
arbol_completo = DecisionTreeClassifier(random_state=444)
path = arbol_completo.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas_path = np.unique(path.ccp_alphas)

# El ultimo alpha suele dejar solo el nodo raiz.
if len(ccp_alphas_path) > 1:
    ccp_alphas_path = ccp_alphas_path[:-1]

# Grilla de hasta 50 valores distribuida sobre todo el pruning path.
n_alpha = min(50, len(ccp_alphas_path))
indices_alpha = np.unique(
    np.round(
        np.linspace(0, len(ccp_alphas_path) - 1, n_alpha)
    ).astype(int)
)
grilla_alpha = ccp_alphas_path[indices_alpha]

# 3. 10-fold cross-validation sobre TRAIN exclusivamente
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=444)

busqueda_alpha = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=444),
    param_grid={"ccp_alpha": grilla_alpha},
    scoring="accuracy",
    cv=cv10,
    n_jobs=-1,
    return_train_score=True,
)

busqueda_alpha.fit(X_train, y_train)

alpha_optimo = float(busqueda_alpha.best_params_["ccp_alpha"])
accuracy_cv_optima = float(busqueda_alpha.best_score_)
error_cv_optimo = 1.0 - accuracy_cv_optima

print("\n" + "=" * 76)
print("INCISO 4 - SELECCION DE ccp_alpha")
print("=" * 76)
print("Valores en pruning path:", len(ccp_alphas_path))
print("Valores evaluados por CV:", len(grilla_alpha))
print("ccp_alpha optimo:", f"{alpha_optimo:.10f}")
print("Accuracy promedio 10-fold CV:", f"{accuracy_cv_optima:.4f}")
print("Error promedio 10-fold CV:", f"{error_cv_optimo:.4f}")

# 4. Resultados de CV
resultados_cv = pd.DataFrame(busqueda_alpha.cv_results_)
tabla_cv = resultados_cv[
    ["param_ccp_alpha", "mean_test_score", "std_test_score", "mean_train_score"]
].rename(
    columns={
        "param_ccp_alpha": "ccp_alpha",
        "mean_test_score": "accuracy_validacion",
        "std_test_score": "sd_accuracy_validacion",
        "mean_train_score": "accuracy_train",
    }
).copy()

tabla_cv["ccp_alpha"] = tabla_cv["ccp_alpha"].astype(float)
tabla_cv["error_validacion"] = 1.0 - tabla_cv["accuracy_validacion"]
tabla_cv["error_train"] = 1.0 - tabla_cv["accuracy_train"]
tabla_cv = tabla_cv.sort_values("ccp_alpha").reset_index(drop=True)

tabla_cv.to_csv(
    RUTA_SALIDA / "inciso4_cv_ccp_alpha.csv",
    index=False,
    encoding="utf-8-sig",
)

# Figura 4: error de clasificacion vs ccp_alpha
fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.plot(
    tabla_cv["ccp_alpha"],
    tabla_cv["error_validacion"],
    marker="o",
    markersize=3.5,
    linewidth=1.1,
)
ax.axvline(
    alpha_optimo,
    linestyle="--",
    linewidth=1.0,
    label=rf"$ccp\_alpha^{{CV}}={alpha_optimo:.6g}$",
)
positivos = tabla_cv.loc[tabla_cv["ccp_alpha"] > 0, "ccp_alpha"]
if len(positivos) > 0:
    ax.set_xscale("symlog", linthresh=float(positivos.min()) / 2)
    ax.set_xlim(left=0)
ax.set_xlabel("Costo de complejidad (ccp_alpha)")
ax.set_ylabel("Error de clasificacion (1 - accuracy)")
ax.set_title("Error de clasificacion por validacion cruzada")
ax.legend(frameon=False)
ax.grid(alpha=0.20)
fig.tight_layout()
fig.savefig(
    RUTA_SALIDA / "Figura4_CART_CV.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

# 5. Arbol final con poda optima
arbol_podado = DecisionTreeClassifier(
    random_state=444,
    ccp_alpha=alpha_optimo,
)
arbol_podado.fit(X_train, y_train)

profundidad = int(arbol_podado.get_depth())
n_hojas = int(arbol_podado.get_n_leaves())
n_nodos = int(arbol_podado.tree_.node_count)

print("\nArbol podado final")
print("Profundidad:", profundidad)
print("Hojas:", n_hojas)
print("Nodos:", n_nodos)


# =============================================================================
# INCISO 5
# ARBOL PODADO, IMPORTANCIA Y COMPARACION CON LASSO
# =============================================================================

# 6. Importancia individual de las 68 columnas transformadas
importancia = (
    pd.DataFrame(
        {
            "variable": variables_x,
            "importancia": arbol_podado.feature_importances_,
        }
    )
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
importancia["ranking"] = np.arange(1, len(importancia) + 1)
importancia.to_csv(
    RUTA_SALIDA / "inciso5_importancia_68_predictores.csv",
    index=False,
    encoding="utf-8-sig",
)

n_importancia_cero = int(np.isclose(importancia["importancia"], 0.0).sum())

# 7. Agrupar dummies segun variable original para facilitar interpretacion
variables_directas = {
    "CH06",
    "edad2",
    "nhogar",
    "prop_ocupados",
    "log_ingreso",
    "ingreso_no_declara",
    "IX_MEN10",
}

prefijos = [
    "tamano_emp",
    "NIVEL_ED",
    "MAS_500",
    "PP04A",
    "INTENSI",
    "REGION",
    "CH03",
    "CH04",
    "CH07",
    "CH15",
    "CH16",
    "II7",
]

def variable_original(nombre):
    if nombre in variables_directas:
        return nombre
    for pref in prefijos:
        if nombre.startswith(pref + "_"):
            return pref
    return nombre.split("_")[0]

importancia["variable_original"] = importancia["variable"].map(variable_original)
importancia_agrupada = (
    importancia.groupby("variable_original", as_index=False)["importancia"]
    .sum()
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
importancia_agrupada["porcentaje"] = 100 * importancia_agrupada["importancia"]
importancia_agrupada.to_csv(
    RUTA_SALIDA / "inciso5_importancia_agrupada.csv",
    index=False,
    encoding="utf-8-sig",
)

# 8. Primera regla del arbol
indice_raiz = int(arbol_podado.tree_.feature[0])
variable_raiz = variables_x[indice_raiz]
umbral_raiz = float(arbol_podado.tree_.threshold[0])
ingreso_equivalente = (
    float(np.expm1(umbral_raiz)) if variable_raiz == "log_ingreso" else np.nan
)

# Etiquetas descriptivas para que el grafico sea consistente con el Cuadro 1 de Parte I.
def etiqueta_amigable(nombre):
    mapa_directo = {
        "log_ingreso": "Log(1 + ingreso principal)",
        "CH06": "Edad",
        "edad2": "Edad al cuadrado",
        "nhogar": "Integrantes del hogar",
        "prop_ocupados": "Proporcion de ocupados del hogar",
        "ingreso_no_declara": "Ingreso principal no declarado",
        "IX_MEN10": "Menores de 10 anos en el hogar",
        "CH04_2": "Mujer",
        "PP04A_2.0": "Establecimiento: privado",
        "PP04A_3.0": "Establecimiento: otro tipo",
        "NIVEL_ED_2": "Educacion: primario completo",
        "NIVEL_ED_3": "Educacion: secundario incompleto",
        "NIVEL_ED_4": "Educacion: secundario completo",
        "NIVEL_ED_5": "Educacion: superior/univ. incompleto",
        "NIVEL_ED_6": "Educacion: superior/univ. completo",
        "NIVEL_ED_7": "Educacion: sin instruccion",
        "REGION_40": "Region: Noroeste",
        "REGION_41": "Region: Noreste",
        "REGION_42": "Region: Cuyo",
        "REGION_43": "Region: Pampeana",
        "REGION_44": "Region: Patagonia",
    }
    return mapa_directo.get(nombre, nombre)

mapa_original = {
    "log_ingreso": "Log(1 + ingreso principal)",
    "PP04A": "Tipo de establecimiento",
    "NIVEL_ED": "Nivel educativo",
    "tamano_emp": "Tamano del establecimiento",
    "CH06": "Edad",
    "CH04": "Sexo",
    "edad2": "Edad al cuadrado",
    "INTENSI": "Intensidad laboral",
    "REGION": "Region",
    "nhogar": "Integrantes del hogar",
    "ingreso_no_declara": "Ingreso principal no declarado",
    "CH07": "Estado civil",
    "II7": "Tenencia de la vivienda",
    "CH15": "Lugar de nacimiento",
    "prop_ocupados": "Proporcion de ocupados del hogar",
    "IX_MEN10": "Menores de 10 anos en el hogar",
    "CH16": "Residencia hace 5 anos",
    "CH03": "Parentesco",
    "MAS_500": "Aglomerado de 500 mil o mas",
}

# 9. Figura 5: Panel A arbol / Panel B importancia agrupada
# Para el Panel A se muestran los primeros niveles del MISMO arbol podado.
fig = plt.figure(figsize=(11, 13.5))
gs = fig.add_gridspec(2, 1, height_ratios=[1.05, 1.0], hspace=0.28)

ax1 = fig.add_subplot(gs[0])
plot_tree(
    arbol_podado,
    feature_names=[etiqueta_amigable(v) for v in variables_x],
    class_names=["Formal", "Informal"],
    max_depth=2,
    rounded=True,
    fontsize=7,
    ax=ax1,
)
ax1.set_title("Panel A. Arbol CART podado (primeros niveles)", fontsize=11)

ax2 = fig.add_subplot(gs[1])
graf_imp = importancia_agrupada.copy()
graf_imp["etiqueta"] = graf_imp["variable_original"].map(mapa_original).fillna(graf_imp["variable_original"])
graf_imp = graf_imp.sort_values("importancia", ascending=True)
ax2.barh(graf_imp["etiqueta"], graf_imp["importancia"])
ax2.set_xlabel("Importancia acumulada")
ax2.set_ylabel("")
ax2.set_title("Panel B. Importancia por variable original", fontsize=11)
ax2.tick_params(axis="y", labelsize=8)
ax2.grid(axis="x", alpha=0.18)

fig.tight_layout()
fig.savefig(
    RUTA_SALIDA / "Figura5_CART_paneles.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

# 10. Comparacion empirica con LASSO usando el resultado del Cuadro 1 de Parte I
# Variables que LASSO llevo exactamente a cero:
variables_lasso_cero = {
    "IX_MEN10": "Menores de 10 anos en el hogar",
    "CH03_3": "Parentesco: hijo/a o hijastro/a",
    "CH03_5": "Parentesco: nieto/a",
    "CH03_9": "Parentesco: otro familiar",
    "NIVEL_ED_2": "Nivel educativo: primario completo",
    "tamano_emp_tam_2": "Tamano del establecimiento: 2 personas",
}

comp_lasso_cart = pd.DataFrame(
    {
        "variable": list(variables_lasso_cero.keys()),
        "descripcion": list(variables_lasso_cero.values()),
    }
)
comp_lasso_cart = comp_lasso_cart.merge(
    importancia[["variable", "importancia"]],
    on="variable",
    how="left",
)

if comp_lasso_cart["importancia"].isna().any():
    faltan = comp_lasso_cart.loc[
        comp_lasso_cart["importancia"].isna(), "variable"
    ].tolist()
    raise ValueError(f"No se encontraron en X las variables LASSO: {faltan}")

comp_lasso_cart["cart_cero"] = np.isclose(
    comp_lasso_cart["importancia"], 0.0, atol=1e-15
)
comp_lasso_cart["coinciden"] = comp_lasso_cart["cart_cero"]

n_lasso_cero = len(comp_lasso_cart)
n_coinciden = int(comp_lasso_cart["coinciden"].sum())
prop_lasso_tambien_cart = n_coinciden / n_lasso_cero
prop_cart_tambien_lasso = n_coinciden / n_importancia_cero

comp_lasso_cart.to_csv(
    RUTA_SALIDA / "Cuadro2_CART_vs_LASSO.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 76)
print("INCISO 5 - COMPARACION CART vs LASSO")
print("=" * 76)
print(comp_lasso_cart.to_string(index=False))
print(f"\nLASSO llevo a cero: {n_lasso_cero} predictores")
print(f"CART asigno importancia cero: {n_importancia_cero} predictores")
print(f"Coinciden en cero: {n_coinciden}")
print(
    "Proporcion de variables eliminadas por LASSO que CART tambien ignora:",
    f"{100 * prop_lasso_tambien_cart:.1f}%",
)
print(
    "Proporcion de variables ignoradas por CART que LASSO tambien llevo a cero:",
    f"{100 * prop_cart_tambien_lasso:.1f}%",
)

# 11. Guardar modelo y metadatos para Parte III
joblib.dump(arbol_podado, RUTA_SALIDA / "arbol_CART_final.joblib")

pd.Series(variables_x, name="variable").to_csv(
    RUTA_SALIDA / "variables_CART.csv",
    index=False,
    encoding="utf-8-sig",
)

resumen = {
    "n_train": int(X_train.shape[0]),
    "n_test": int(X_test.shape[0]),
    "n_predictores": int(X_train.shape[1]),
    "ccp_alpha_optimo": alpha_optimo,
    "accuracy_cv_10fold": accuracy_cv_optima,
    "error_cv_10fold": error_cv_optimo,
    "profundidad": profundidad,
    "numero_hojas": n_hojas,
    "numero_nodos": n_nodos,
    "variable_raiz": variable_raiz,
    "umbral_raiz": umbral_raiz,
    "ingreso_equivalente_si_log_ingreso": ingreso_equivalente,
    "predictores_importancia_cero": n_importancia_cero,
    "lasso_predictores_cero": n_lasso_cero,
    "coincidencias_lasso_cart_cero": n_coinciden,
    "prop_lasso_cero_tambien_cart_cero": prop_lasso_tambien_cart,
    "prop_cart_cero_tambien_lasso_cero": prop_cart_tambien_lasso,
    "nota_parte_3": (
        "El test no fue utilizado en Parte II. En Parte III se debe cargar este "
        "mismo arbol y evaluarlo sobre el mismo X_test que Logit, KNN, LASSO y Ridge."
    ),
}

with open(RUTA_SALIDA / "resumen_CART_ParteII.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)

# 12. Resumen en consola
print("\n" + "=" * 76)
print("RESUMEN FINAL PARTE II")
print("=" * 76)
print(f"ccp_alpha optimo: {alpha_optimo:.10f}")
print(f"Accuracy CV: {accuracy_cv_optima:.4f}")
print(f"Error CV: {error_cv_optimo:.4f}")
print(f"Profundidad: {profundidad}; hojas: {n_hojas}; nodos: {n_nodos}")
print(f"Raiz: {variable_raiz} <= {umbral_raiz:.6f}")
if variable_raiz == "log_ingreso":
    print(f"Ingreso aproximado equivalente: {ingreso_equivalente:,.0f}")
print(f"Importancia cero en CART: {n_importancia_cero}/{len(importancia)}")
print(f"Coincidencia CART-LASSO: {n_coinciden}/{n_lasso_cero} = {100*prop_lasso_tambien_cart:.1f}%")
print("\nIMPORTANTE PARA PARTE III:")
print("- No volver a seleccionar ccp_alpha con test.")
print("- Cargar arbol_CART_final.joblib.")
print("- Usar exactamente el mismo X_test/y_test para los cinco modelos.")
print("- Calcular matriz de confusion (p > 0.5), ROC, metricas del TP1 y 1-accuracy.")
print("\nArchivos guardados en:", RUTA_SALIDA.resolve())


BASE Y PARTICION
X_train: (13692, 68)
X_test : (5868, 68)
Predictores: 68
Informalidad train: 0.456
Informalidad test : 0.4352

INCISO 4 - SELECCION DE ccp_alpha
Valores en pruning path: 662
Valores evaluados por CV: 50
ccp_alpha optimo: 0.0003998397
Accuracy promedio 10-fold CV: 0.8013
Error promedio 10-fold CV: 0.1987

Arbol podado final
Profundidad: 12
Hojas: 51
Nodos: 101


C:\Users\JOEL\AppData\Local\Temp\ipykernel_7160\3753180548.py:346: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



INCISO 5 - COMPARACION CART vs LASSO
        variable                            descripcion  importancia  cart_cero  coinciden
        IX_MEN10         Menores de 10 anos en el hogar     0.000000       True       True
          CH03_3        Parentesco: hijo/a o hijastro/a     0.000000       True       True
          CH03_5                    Parentesco: nieto/a     0.000000       True       True
          CH03_9              Parentesco: otro familiar     0.000000       True       True
      NIVEL_ED_2     Nivel educativo: primario completo     0.000000       True       True
tamano_emp_tam_2 Tamano del establecimiento: 2 personas     0.003341      False      False

LASSO llevo a cero: 6 predictores
CART asigno importancia cero: 45 predictores
Coinciden en cero: 5
Proporcion de variables eliminadas por LASSO que CART tambien ignora: 83.3%
Proporcion de variables ignoradas por CART que LASSO tambien llevo a cero: 11.1%

RESUMEN FINAL PARTE II
ccp_alpha optimo: 0.0003998397
Accuracy CV: